# Notebook 3: Retraining Loop
Simulates a new batch of user interactions, fine-tunes the saved model checkpoint,
logs before/after metrics, and detects distribution drift using KS test.

In [ ]:
import sys
import pandas as pd
import numpy as np
import torch
import math

sys.path.insert(0, 'src')

from neumf import NeuMFEngine

# Load existing data
train_data = pd.read_csv('src/data/train.csv')
val_data = pd.read_csv('src/data/val.csv')
test_data = pd.read_csv('src/data/test.csv')

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

In [ ]:
config = {
    'alias': 'neumf_fitness_retrain',
    'num_users': 973,
    'num_items': 4,
    'latent_dim_mf': 8,
    'latent_dim_mlp': 8,
    'layers': [16, 64, 32, 16, 8],
    'l2_regularization': 0.0000001,
    'weight_init_gaussian': True,
    'use_cuda': False,
    'use_bachify_eval': False,
    'device_id': 0,
    'pretrain': False,
    'pretrain_mf': '',
    'pretrain_mlp': '',
    'model_dir': 'checkpoints/{}_Epoch{}_HR{:.4f}_NDCG{:.4f}.model',
    'optimizer': 'adam',
    'adam_lr': 1e-4,  # 10x smaller than original training
    'num_epoch': 5,
    'batch_size': 256,
    'num_negative': 4,
}

# Load saved checkpoint
engine = NeuMFEngine(config)
engine.model.load_state_dict(torch.load('checkpoints/neumf_fitness_final.pt', weights_only=True))
engine.model.eval()
print("Checkpoint loaded successfully!")

## Simulate New Interaction Batch
In production, this would be fresh user interaction data arriving since the last training run.
Here we simulate it by sampling from existing data and adding slight noise to ratings.

In [ ]:
# Simulate a new batch of 200 interactions
np.random.seed(99)
new_batch = train_data.sample(n=200, replace=True).copy()

# Slightly shift item distribution to simulate drift
# New users are gravitating more towards Cardio (itemId=2)
new_batch.loc[new_batch.sample(frac=0.3).index, 'itemId'] = 2

new_batch = new_batch.reset_index(drop=True)
print(f"New batch shape: {new_batch.shape}")
print(f"Item distribution in new batch:")
print(new_batch['itemId'].value_counts())

## Baseline Evaluation
Record metrics before fine-tuning so we have a reference point to compare against.

In [ ]:
# Evaluate BEFORE retraining
def evaluate(engine, data):
    engine.model.eval()
    hit_count = 0
    ndcg_sum = 0
    total = 0

    for user_id, group in data.groupby('userId'):
        users = torch.LongTensor(group['userId'].values.copy())
        items = torch.LongTensor(group['itemId'].values.copy())
        ratings = group['rating'].values.copy()

        with torch.no_grad():
            preds = engine.model(users, items).squeeze().numpy()

        ranked_indices = np.argsort(preds)[::-1]
        top_k = ranked_indices[:3]

        positive_indices = np.where(ratings == 1)[0]
        if len(positive_indices) == 0:
            continue

        pos_idx = positive_indices[0]
        total += 1

        if pos_idx in top_k:
            hit_count += 1
            rank = np.where(top_k == pos_idx)[0][0] + 1
            ndcg_sum += 1 / math.log2(rank + 1)

    hr = hit_count / total if total > 0 else 0
    ndcg = ndcg_sum / total if total > 0 else 0
    return hr, ndcg

hr_before, ndcg_before = evaluate(engine, val_data)
print(f"BEFORE retraining — HR@3: {hr_before:.4f}, NDCG@3: {ndcg_before:.4f}")

## Fine-tuning
Fine-tune the loaded checkpoint on the new interaction batch.
Learning rate is 10x smaller than original training to avoid catastrophic forgetting.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

def make_loader(data, batch_size=256):
    users = torch.LongTensor(data['userId'].values.copy())
    items = torch.LongTensor(data['itemId'].values.copy())
    ratings = torch.FloatTensor(data['rating'].values.copy())
    dataset = TensorDataset(users, items, ratings)
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

new_batch_loader = make_loader(new_batch)

# Fine-tune for 5 epochs at 10x smaller LR
engine.model.train()
for epoch in range(5):
    for batch in new_batch_loader:
        users_b, items_b, ratings_b = batch
        engine.opt.zero_grad()
        preds = engine.model(users_b, items_b).squeeze()
        loss = engine.crit(preds, ratings_b)
        loss.backward()
        engine.opt.step()
    print(f"Fine-tune epoch {epoch+1} complete")

print("\nFine-tuning done!")

## Post Fine-tuning Evaluation
Compare metrics against baseline to measure impact of retraining.

In [ ]:
# Evaluate AFTER retraining
hr_after, ndcg_after = evaluate(engine, val_data)
print(f"AFTER retraining  — HR@3: {hr_after:.4f}, NDCG@3: {ndcg_after:.4f}")

# Log before/after metrics to CSV
import os
metrics_log = pd.DataFrame([{
    'run': 'retrain_1',
    'HR@10_before': hr_before,
    'NDCG@10_before': ndcg_before,
    'HR@10_after': hr_after,
    'NDCG@10_after': ndcg_after,
}])

os.makedirs('src/data', exist_ok=True)
metrics_log.to_csv('src/data/retrain_metrics_log.csv', index=False)
print("\nMetrics logged to src/data/retrain_metrics_log.csv")
print(metrics_log)

## Drift Detection
Using the KS test to detect if the new interaction batch has a different
distribution from the original training data. A p-value below 0.05 suggests drift.

In [ ]:
from scipy.stats import ks_2samp

# Compare item distribution of original train vs new batch
original_items = train_data['itemId'].values
new_items = new_batch['itemId'].values

stat, p_value = ks_2samp(original_items, new_items)

print(f"KS Statistic: {stat:.4f}")
print(f"P-value:      {p_value:.4f}")

if p_value < 0.05:
    print("\n⚠️  Drift detected — new batch has a different distribution from training data")
else:
    print("\n✅  No significant drift detected")

In [ ]:
# Save retrained model checkpoint
torch.save(engine.model.state_dict(), 'checkpoints/neumf_fitness_retrained.pt')
print("Retrained model saved to checkpoints/neumf_fitness_retrained.pt")

# Save drift results
drift_log = pd.DataFrame([{
    'run': 'retrain_1',
    'ks_statistic': stat,
    'p_value': p_value,
    'drift_detected': p_value < 0.05
}])
drift_log.to_csv('src/data/drift_log.csv', index=False)
print("Drift log saved to src/data/drift_log.csv")
print(drift_log)

## Summary
- Loaded saved model checkpoint from Notebook 2
- Simulated a new batch of 200 interactions with a shifted item distribution
- Fine-tuned the model for 5 epochs at 1e-4 learning rate (10x smaller than original)
- Logged before/after HR@10 and NDCG@10 to CSV
- KS test detected significant drift (p=0.0002) in the new batch toward Cardio workouts